# Preparing Training Data

First of all, we need to convert the training data from .json to .spacy (so that we can train our model)

In [ ]:
import re
import spacy
from spacy.tokenizer import Tokenizer

prefix_re = re.compile(r'[(\[\'"]|<i>')
suffix_re = re.compile(r'[.,;:?!)\]\'"]|</i>[.,;:?!)\]\'"]*')
infix_re = re.compile(r'[-/+]')

def custom_tokenizer(nlp):
    return Tokenizer(
        nlp.vocab,
        prefix_search=prefix_re.search,
        suffix_search=suffix_re.search,
        infix_finditer=infix_re.finditer
    )

In [ ]:
import json
from spacy.tokens import DocBin

json_files = [
    'gutbrainie2025/Annotations/Train/platinum_quality/json_format/train_platinum.json',
    'gutbrainie2025/Annotations/Train/gold_quality/json_format/train_gold.json',
    'gutbrainie2025/Annotations/Train/silver_quality/json_format/train_silver.json',
    # 'gutbrainie2025/Annotations/Train/bronze_quality/json_format/train_bronze.json'
]

nlp = spacy.blank('en')
nlp.tokenizer = custom_tokenizer(nlp)
db = DocBin()

for json_file in json_files:
    counter = 0
    print('Parsing {}'.format(json_file))
    f = open(json_file)
    data = json.load(f)

    for article in data:
        title = data[article]['metadata']['title']
        abstract = data[article]['metadata']['abstract']
        entities = data[article]['entities']

        title_doc = nlp(title)
        abstract_doc = nlp(abstract)

        title_ents = []
        abstract_ents = []
        for entity in entities:
            start = int(entity['start_idx'])
            end = int(entity['end_idx']) + 1
            label = entity['label']
            
            if entity['location'] == 'title':
                span = title_doc.char_span(start, end, label=label, alignment_mode='strict')
                if span:
                    title_ents.append(span)
                else:
                    print('-'*60)
                    print(title)
                    print('TITLE TOKENS -> ', [token for token in title_doc])
                    print('SPAN NOT FOUND -> ', title_doc.text[start:end])
            elif entity['location'] == 'abstract':
                span = abstract_doc.char_span(start, end, label=label, alignment_mode='strict') # with alignment_mode='strict' only exact matches
                if span and span.text != 'D': # to remove D span, which is in conflict with Vitamin D
                    abstract_ents.append(span)
                else:
                    print('-'*60)
                    print(title)
                    print('ABSTRACT TOKENS -> ', [token for token in abstract_doc])
                    print('SPAN NOT FOUND -> ', abstract_doc.text[start:end])
            else:
                print('ERROR: {}'.format(entity['location']))

        title_doc.ents = title_ents
        abstract_doc.ents = abstract_ents
        db.add(title_doc)
        db.add(abstract_doc)
        counter += 1
    print(counter)

#db.to_disk('./train.spacy')
                